In [9]:
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scipy as sp
from pathlib import Path
import sys
import harmonypy as hm
sys.path.insert(0, "..")
import utils as ut

In [4]:
QCDir = Path("../../result/qc/qc.h5ad")
adata = sc.read_h5ad(QCDir)

In [ ]:
# ── Batch-aware HVG selection (Pearson residuals, crc-atlas pattern) ──
# seurat_v3: Pearson residuals on raw counts, robust to count distribution differences
# batch_key="Study": compute variance within each study, then rank across studies
# layer="counts": uses raw integer counts (no normalization before this step)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    layer="counts",
    batch_key="Study",
    subset=False,
)

n_hvg = adata.var["highly_variable"].sum()
print(f"HVGs selected: {n_hvg} / {adata.n_vars}")

In [ ]:
# ── Subset to HVGs + normalize ──
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
print(f"HVG subset: {adata_hvg.shape}")

# Normalize + log1p AFTER HVG selection
# At this point zeros are real low expression, not outer-join artifacts
sc.pp.normalize_total(adata_hvg, target_sum=1e4)
sc.pp.log1p(adata_hvg)
print("Normalization + log1p done")

In [ ]:
# ── PCA on HVG subset ──
sc.pp.pca(adata_hvg, n_comps=50, svd_solver="arpack")
print(f"PCA done: {adata_hvg.obsm['X_pca'].shape}")

# Pre-Harmony PCA overview
sc.pl.pca(adata_hvg, color="Study", title="PCA (pre-Harmony)")

In [ ]:
# ── Harmony on "Study" only ──
# Tech (scRNA/snRNA) is nested in Study, cannot be independently estimated
pcs = adata_hvg.obsm["X_pca"]
print(f"Input PCs: {pcs.shape}")

harmony_out = hm.run_harmony(pcs, adata_hvg.obs, "Study")
adata_hvg.obsm["X_pca_harmony"] = harmony_out.Z_corr

# Post-Harmony PCA
sc.pl.pca(adata_hvg, color="Study", title="PCA (post-Harmony)")

# UMAP + Leiden on Harmony-corrected PCs
sc.pp.neighbors(adata_hvg, use_rep="X_pca_harmony", n_neighbors=15)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg, key_added="leiden", flavor="igraph", n_iterations=2, directed=False)
print(f"Leiden clusters: {adata_hvg.obs['leiden'].nunique()}")

In [ ]:
# ── Evaluation plots ──
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
sc.pl.umap(adata_hvg, color="Study", ax=axes[0, 0], show=False, title="Study", palette="Set1")
sc.pl.umap(adata_hvg, color="Tissue", ax=axes[0, 1], show=False, title="Tissue", palette="Set2")
sc.pl.umap(adata_hvg, color="Tech", ax=axes[1, 0], show=False, title="Tech", palette="Set1")
sc.pl.umap(adata_hvg, color="leiden", ax=axes[1, 1], show=False,
           title=f"Leiden ({adata_hvg.obs['leiden'].nunique()} clusters)",
           legend_loc="on data", legend_fontsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# ── Transfer coords back to full adata + save ──
# Transfer Harmony-corrected PCs and UMAP to full AnnData (same cells, all genes)
common_idx = adata.obs_names.isin(adata_hvg.obs_names)
adata.obsm["X_pca_harmony"] = np.full((adata.n_obs, adata_hvg.obsm["X_pca_harmony"].shape[1]), np.nan)
adata.obsm["X_pca_harmony"][common_idx, :] = adata_hvg.obsm["X_pca_harmony"]

adata.obsm["X_umap"] = np.full((adata.n_obs, 2), np.nan)
adata.obsm["X_umap"][common_idx, :] = adata_hvg.obsm["X_umap"]

adata.obs["leiden"] = "NA"
adata.obs.loc[common_idx, "leiden"] = adata_hvg.obs["leiden"].values

print(f"Transferred coords to full adata: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# Save
outdir = Path("../../result/harmony")
outdir.mkdir(parents=True, exist_ok=True)
adata_hvg.write_h5ad(outdir / "harmony_hvg.h5ad", compression="lzf")
adata.write_h5ad(outdir / "harmony_full.h5ad", compression="lzf")
print(f"Saved: {outdir / 'harmony_hvg.h5ad'}  ({adata_hvg.n_vars} HVG genes)")
print(f"Saved: {outdir / 'harmony_full.h5ad'}  ({adata.n_vars} genes, UMAP + Harmony coords)")